Basically herw we study how we can performa a hybrid search using both sementic search and key word search when retriving from a vector data base. for that we need to maintain two embeddings for sementic search and key word search. for sementic search we can generate the dense vector using a embedding like openAI embedding. for the key word search we can build sparse vector using a embedding like one hot encodding.

In [1]:
#install required packages 
!pip install numpy scikit-learn sentence-transformers -q


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#we define 4 docuemnts (in form of data chunks)
documents=["Michael Jackson (1958-2009) was an American singer, songwriter, and dancer dubbed the King of Pop,widely regarded as one of the most significant cultural figures of the 20th century. He revolutionized music videos, popularised dance moves like the moonwalk, and sold over 500 million records, with Thriller remaining the best-selling album in history",
          "As of May 2026, the Iran-Israel conflict has escalated into a direct, large-scale war, moving beyond proxy battles to direct air strikes. Coordinated U.S.-Israel operations against Iran intensified in early 2026, following the April 2024 shift in hostilities, resulting in significant infrastructure damage, widespread fighting, and high civilian casualties",
           "Agentic AI refers to autonomous systems that act as intelligent agents to achieve complex, long-term goals with limited human supervision. Unlike generative AI that responds to prompts, agentic AI uses reasoning, planning, and tool use to execute multi-step workflows independently, often acting as a digital employee.",
           "Researchers are professionals who conduct systematic, thorough investigations to discover new knowledge, solve problems, or create new concepts across fields like science, medicine, and social sciences. They design studies, collect data via experiments or surveys, and analyze findings to publish reports or inform decisions."]

In [3]:
#preprocessing the text, removing the puncutation marks, make all words lowercase etc

#import built-in regular expression module, sing for pattern matching and text searching
import re

#function for preprocessing the document content
def preprocessing(text):
    text=text.lower()
    text=re.sub(r'[^\w\s]','',text)
    return text

preprocessed_documents=[preprocessing(doc) for doc in documents]

for doc in preprocessed_documents:
    print(doc)


michael jackson 19582009 was an american singer songwriter and dancer dubbed the king of popwidely regarded as one of the most significant cultural figures of the 20th century he revolutionized music videos popularised dance moves like the moonwalk and sold over 500 million records with thriller remaining the bestselling album in history
as of may 2026 the iranisrael conflict has escalated into a direct largescale war moving beyond proxy battles to direct air strikes coordinated usisrael operations against iran intensified in early 2026 following the april 2024 shift in hostilities resulting in significant infrastructure damage widespread fighting and high civilian casualties
agentic ai refers to autonomous systems that act as intelligent agents to achieve complex longterm goals with limited human supervision unlike generative ai that responds to prompts agentic ai uses reasoning planning and tool use to execute multistep workflows independently often acting as a digital employee
resea

In [4]:
test_query="agentic AI is a subset of generative AI"

Using keyword search to find the best matching document(data chunk) for the given test_query

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer #call the TFid method to do the key word search, (another method like one hot encoding)
from sklearn.metrics.pairwise import cosine_similarity #to check the similarity
import numpy as np

In [6]:
vectorizer=TfidfVectorizer()
sparse_vectors=vectorizer.fit_transform(preprocessed_documents) #arange all four doucments as sparse vectors

In [7]:
sparse_vectors.toarray() #provide 4 sparse vectors

array([[0.12688026, 0.        , 0.        , 0.12688026, 0.12688026,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.12688026,
        0.12688026, 0.12688026, 0.        , 0.13242276, 0.        ,
        0.        , 0.08098602, 0.        , 0.        , 0.12688026,
        0.        , 0.        , 0.12688026, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.12688026, 0.        , 0.12688026, 0.12688026,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.12688026, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.12688026,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.12688026, 0.        , 0.12688026, 0.        , 0.        ,
        0.10003385, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.  

In [8]:
len(sparse_vectors.toarray()) #length of sparse array equal to no of documents we given

4

In [9]:
#Check the length of a single sparse vector, it is the size of the vocabulary of the document, thi would be same for all documents(data chunks) as well as test query
len(sparse_vectors.toarray()[3])

151

In [10]:
#arrange test query also as a sparse vector
sparse_test_query=vectorizer.transform([test_query])#since need to make with similar vocabulary, use transform function instead of fit_transform
len(sparse_test_query.toarray()[0])

151

In [11]:
#Now to see the keyword similarity between the test query and documents, we use the cosinesimilarity function
keyword_similarity=cosine_similarity(sparse_vectors,sparse_test_query)

In [ ]:
keyword_similarity#3rd data chunk gives higerst similarity value

array([[0.09194751],
       [0.0333667 ],
       [0.46720126],
       [0.        ]])

In [13]:
#ranking according to the similarity
ranked_indexes=np.argsort(keyword_similarity,axis=0)[::-1].flatten()
ranked_indexes

array([2, 0, 1, 3])

In [16]:
ranked_documents=[documents[i] for i in ranked_indexes]
for doc in ranked_documents:
    print (doc)

Agentic AI refers to autonomous systems that act as intelligent agents to achieve complex, long-term goals with limited human supervision. Unlike generative AI that responds to prompts, agentic AI uses reasoning, planning, and tool use to execute multi-step workflows independently, often acting as a digital employee.
Michael Jackson (1958-2009) was an American singer, songwriter, and dancer dubbed the King of Pop,widely regarded as one of the most significant cultural figures of the 20th century. He revolutionized music videos, popularised dance moves like the moonwalk, and sold over 500 million records, with Thriller remaining the best-selling album in history
As of May 2026, the Iran-Israel conflict has escalated into a direct, large-scale war, moving beyond proxy battles to direct air strikes. Coordinated U.S.-Israel operations against Iran intensified in early 2026, following the April 2024 shift in hostilities, resulting in significant infrastructure damage, widespread fighting, a

Let us consider how we can use the sementic Search for the same process 

In [20]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np

In [21]:
#creating dense vector using a embedding model
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
dense_vectors=embedding_model.encode(preprocessed_documents)
len(dense_vectors[0])


c:\Users\shara\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shara\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10806.55it/s]


384

In [24]:
dense_test_query=embedding_model.encode([test_query])
len(dense_test_query[0])

384

In [25]:
dense_vectors_similarity=cosine_similarity(dense_vectors,dense_test_query)

In [26]:
ranked_indexes_2=np.argsort(dense_vectors_similarity,axis=0)[::-1].flatten()
ranked_indexes

array([2, 0, 1, 3])

In [28]:
ranked_documents=[documents[i] for i in ranked_indexes_2]
for doc in ranked_documents:
    print (doc)

Agentic AI refers to autonomous systems that act as intelligent agents to achieve complex, long-term goals with limited human supervision. Unlike generative AI that responds to prompts, agentic AI uses reasoning, planning, and tool use to execute multi-step workflows independently, often acting as a digital employee.
Michael Jackson (1958-2009) was an American singer, songwriter, and dancer dubbed the King of Pop,widely regarded as one of the most significant cultural figures of the 20th century. He revolutionized music videos, popularised dance moves like the moonwalk, and sold over 500 million records, with Thriller remaining the best-selling album in history
As of May 2026, the Iran-Israel conflict has escalated into a direct, large-scale war, moving beyond proxy battles to direct air strikes. Coordinated U.S.-Israel operations against Iran intensified in early 2026, following the April 2024 shift in hostilities, resulting in significant infrastructure damage, widespread fighting, a

Now we have to check on how we can combine both keyword search and sementic search using RAG in LanChain

For this, first we create dense score, and sparse score based on eqn 1/R where R means the rank of each data chunck based on their rank.
Then create a hybrid score using a weigted sum of dense search and the sparse search as hybrid_score=alpha.dense_score + (1-alpha).sparse search

In [ ]:
!pip install pypdf -q
!pip install langchain -q
!pip install langchain_community  -q
!pip install langchain_openai -q
!pip install langchain_chroma -q
!pip install rank_bm25 -q# updated version of tfidf for keyword search


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
#call the API key as an environment variable
#to manage API key as a local enviornment variable we need OS library, and to load the env variables from .env file we need to install python-dotenv package
%pip install python-dotenv

import os

#load openai key from .env file, first import the library to load env variables
from dotenv import load_dotenv
from pathlib import Path

env_path=Path.cwd() /'.env'#give the .env file path
print("Path to .env file:", env_path)
print("File exists:", env_path.exists())
# Load environment variables from .env file
load_dotenv(env_path,override=True)


#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
api_key = os.getenv("OPENAI_API_KEY")
print("Loaded:", api_key is not None)

Note: you may need to restart the kernel to use updated packages.
Path to .env file: c:\Users\shara\OneDrive\Documents\Coding Stuff\Generative AI\LangChain\.env
File exists: True
Loaded: True



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
#initialize the chatOPENAI model
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9, openai_api_key=api_key)

In [33]:
#intialize a embedding model for semantic searhc purpose
from langchain_openai import OpenAIEmbeddings 
embedding=OpenAIEmbeddings(model='text-embedding-3-small')

In [34]:
from langchain_community.document_loaders import PyPDFLoader

#initialize the pdf document loader
loader=PyPDFLoader("/Users/shara/OneDrive/Documents/Coding Stuff/Generative AI/LangChain/Documents/example_pdf_file.pdf")
pdf_data=loader.load()

print(pdf_data)

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-06-17T22:17:35+05:30', 'author': 'Dinesh Piyasamara', 'moddate': '2024-06-17T22:17:35+05:30', 'source': '/Users/shara/OneDrive/Documents/Coding Stuff/Generative AI/LangChain/Documents/example_pdf_file.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content="Sri Lanka's national cricket team achieved a historic milestone by winning the ICC Cricket World \nCup in 1996.  \nThe team is known for producing cricket legends like Muttiah Muralitharan, the highest wicket-taker \nin Test cricket history.  \nKumar Sangakkara and Mahela Jayawardene are celebrated for their prolific batting partnerships.  \nSanath Jayasuriya revolutionized one-day cricket with his explosive batting style.  \nThe Galle International Stadium, with its stunning backdrop of the Galle Fort, is one of the world's \nmost picturesque cricket venues.  \nSri Lanka won the ICC 

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

#intialize the text splitter with a chunck size of 50 and no overlap.
#here we use a chachter-level function to count the length of chunks(not the tiktoekn method at this moment)
text_splitter=RecursiveCharacterTextSplitter(chunk_size=50,chunk_overlap=30,length_function=len)#which means we create chunks with chrhactersize of 50.
chunks=text_splitter.split_documents(pdf_data)

In [36]:
len(chunks)

37

Create the Sementic Search Retriver

In [ ]:
from langchain_chroma import Chroma
vectorStore=Chroma.from_documents(chunks, embedding)
vectorstore_retriever=vectorStore.as_retriever(search_kwargs={"k":2})#search_kwargs={"k":2} means most matching two data chunks
vectorstore_retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001BFAAE970E0>, search_kwargs={'k': 2})

Create Keyword Search Retriever

In [39]:
from langchain_community.retrievers import BM25Retriever
keyword_retriever=BM25Retriever.from_documents(chunks)
keyword_retriever.k=2
keyword_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001BFAC81C2F0>, k=2)

Create hybrid search retriever

In [ ]:
#To create hybrid retirever, with the weighting function we use ensemble retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
ensemble_retirever=EnsembleRetriever(retrievers=[vectorstore_retriever,keyword_retriever], weights=[0.5,0.5])#equal weightfor both
ensemble_retirever


EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001BFAAE970E0>, search_kwargs={'k': 2}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001BFAC81C2F0>, k=2)], weights=[0.5, 0.5])

Define the prompt template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

#define a message tempalte for the chatbot
message="""Answer this question using provided context only. {question} Context: {context}"""

#Create a chat prompt template from the message
prompt=ChatPromptTemplate.from_messages([("human",message)])

Cretae the RAG chain

In [53]:
chain=({'context':ensemble_retirever,'question':RunnablePassthrough()}|prompt|llm)

    Invoke the RAG app with a example question

In [60]:
response=chain.invoke('what happen for sri lanka cricekt in 1996')
print(response.content)

Sri Lanka's national cricket team achieved a Cup in 1996.


In [63]:
#check outcomes of each reteriver (sementic, keyword and hybrid)
for doc in vectorstore_retriever.invoke("what happen for sri lanka cricekt in 1996"):
    print (doc.page_content)
    print("-----------------")


Sri Lanka's national cricket team achieved a
-----------------
Cup in 1996.
-----------------


In [64]:
for doc in keyword_retriever.invoke("what happen for sri lanka cricekt in 1996"):
    print (doc.page_content)
    print("-----------------")

for their prolific batting partnerships.
-----------------
in 2014, demonstrating their prowess in the
-----------------


In [65]:
for doc in ensemble_retirever.invoke("what happen for sri lanka cricekt in 1996"):
    print (doc.page_content)
    print("-----------------")

Sri Lanka's national cricket team achieved a
-----------------
for their prolific batting partnerships.
-----------------
Cup in 1996.
-----------------
in 2014, demonstrating their prowess in the
-----------------
